# SoundAQnet — CLI Tutorial

Every step of the pipeline is available as a standalone command-line tool.
This notebook shows the full CLI workflow with shell cells (`!` prefix).

| Command | Purpose |
|---|---|
| `soundaqnet-extract-mel` | Extract log-mel spectrograms → `.npy` |
| `soundaqnet-extract-loudness` | Extract ISO 532-1 loudness → `.npy` |
| `soundaqnet-infer` | Run inference → text result files |
| `soundaqnet-to-df` | Convert result files → CSV / pickle DataFrame |

---

> **Setup** — commands are available after `pip install soundaqnet`.  
> Run cells with `Shift+Enter`.  Replace the example paths with your own.

## 0 · Check installation

In [ ]:
!soundaqnet-extract-mel --help

In [ ]:
!soundaqnet-extract-loudness --help

In [ ]:
!soundaqnet-infer --help

---
## 1 · Step 1 — Extract log-mel spectrograms

```
soundaqnet-extract-mel
    --input_dir   <directory of audio files>
    --output_dir  <directory for .npy outputs>
    --num_workers <parallel threads, default 4>
```

Supported formats: `.wav`, `.mp3`, `.flac`, `.ogg`, `.aiff`, `.m4a`, `.opus`  
Output: one `<stem>.npy` file per clip, shape `(frames, 64)`, dtype `float32`.

In [ ]:
AUDIO_DIR    = "audio/"            # ← replace with your audio directory
MEL_DIR      = "mel_features/"
LOUDNESS_DIR = "loudness_features/"

# Basic extraction (4 worker threads)
!soundaqnet-extract-mel \
    --input_dir  {AUDIO_DIR} \
    --output_dir {MEL_DIR} \
    --num_workers 4

In [ ]:
# Inspect the output
import numpy as np, pathlib

npy_files = sorted(pathlib.Path(MEL_DIR).glob("*.npy"))
print(f"Found {len(npy_files)} mel .npy files")
if npy_files:
    arr = np.load(npy_files[0])
    print(f"  {npy_files[0].name}: shape={arr.shape}, dtype={arr.dtype}")

---
## 2 · Step 2 — Extract ISO 532-1 loudness

```
soundaqnet-extract-loudness
    --input_dir   <directory of audio files>
    --output_dir  <directory for .npy outputs>
    --num_workers <parallel threads, default = cpu_count>
    --target_sr   <48000 | 44100 | 32000>  (default 48000)
    --method      <Varying | Stationary>   (default Varying)
    --sound_field <Free | Diffuse>         (default Free, Windows only)
    --overwrite                             (re-extract already-done files)
```

Output: one `<stem>.npy` file per clip, shape `(T, 1)`, dtype `float32` — total
time-varying loudness in **sone** at 500 Hz (ISO 532-1 standard 2 ms steps).

> **Platform**: Windows uses the bundled `ISO_532-1.exe`; macOS/Linux uses mosqito.

In [ ]:
!soundaqnet-extract-loudness \
    --input_dir  {AUDIO_DIR} \
    --output_dir {LOUDNESS_DIR} \
    --num_workers 4 \
    --target_sr 48000 \
    --method Varying

In [ ]:
npy_loud = sorted(pathlib.Path(LOUDNESS_DIR).glob("*.npy"))
print(f"Found {len(npy_loud)} loudness .npy files")
if npy_loud:
    arr = np.load(npy_loud[0])
    print(f"  {npy_loud[0].name}: shape={arr.shape}, dtype={arr.dtype}")

### Verifying paired files

Each mel `.npy` must have a matching loudness `.npy` with the same stem.

In [ ]:
mel_stems  = {p.stem for p in pathlib.Path(MEL_DIR).glob("*.npy")}
loud_stems = {p.stem for p in pathlib.Path(LOUDNESS_DIR).glob("*.npy")}

missing_loud = mel_stems - loud_stems
missing_mel  = loud_stems - mel_stems

if missing_loud:
    print(f"WARNING: {len(missing_loud)} mel files have no matching loudness: {missing_loud}")
elif missing_mel:
    print(f"WARNING: {len(missing_mel)} loudness files have no matching mel: {missing_mel}")
else:
    print(f"All {len(mel_stems)} files are paired correctly.")

---
## 3 · Step 3 — Run inference

```
soundaqnet-infer
    --dataset_mel           <mel .npy directory>
    --dataset_wav_loudness  <loudness .npy directory>
    --model                 <bundled name or /path/to.pth>  (optional)
    --list-models                                           (show available bundled names)
```

Writes legacy text output files (backward-compatible) to:
- `SoundAQnet_event_probability/`
- `SoundAQnet_scene_ISOPl_ISOEv_PAQ8DAQs/`

In [ ]:
# List available bundled model names
!soundaqnet-infer --list-models

In [ ]:
# Run inference with the default model
!soundaqnet-infer \
    --dataset_mel          {MEL_DIR} \
    --dataset_wav_loudness {LOUDNESS_DIR}

In [ ]:
# Use a specific model variant
!soundaqnet-infer \
    --dataset_mel          {MEL_DIR} \
    --dataset_wav_loudness {LOUDNESS_DIR} \
    --model SoundAQnet_ASC96_AEC95_PAQ1052

In [ ]:
# Check one of the output files
import os

scene_dir = "SoundAQnet_scene_ISOPl_ISOEv_PAQ8DAQs"
if os.path.isdir(scene_dir):
    for f in sorted(os.listdir(scene_dir))[:3]:
        path = os.path.join(scene_dir, f)
        print(f"=== {f} ===")
        with open(path) as fh:
            print(fh.read()[:400])
        print()

---
## 4 · Step 4 — Convert results to a DataFrame

```
soundaqnet-to-df
    --results_dir  <path to SoundAQnet_scene_ISOPl_ISOEv_PAQ8DAQs/>
    --events_dir   <path to SoundAQnet_event_probability/>
    --output       <output CSV path>  (default: soundaqnet_predictions.csv)
```

In [ ]:
!soundaqnet-to-df --help

In [ ]:
!soundaqnet-to-df \
    --results_dir SoundAQnet_scene_ISOPl_ISOEv_PAQ8DAQs \
    --events_dir  SoundAQnet_event_probability \
    --output      soundaqnet_predictions.csv

In [ ]:
import pandas as pd

df = pd.read_csv("soundaqnet_predictions.csv")
print(df.shape)
df.head()

---
## 5 · Full pipeline in one shell script

All four steps combined for scripted/batch use:

In [ ]:
%%bash
set -e

AUDIO_DIR="audio/"
MEL_DIR="mel_features/"
LOUD_DIR="loudness_features/"
WORKERS=4

echo "=== Step 1: Extract mel spectrograms ==="
soundaqnet-extract-mel \
    --input_dir  "$AUDIO_DIR" \
    --output_dir "$MEL_DIR" \
    --num_workers "$WORKERS"

echo "=== Step 2: Extract loudness ==="
soundaqnet-extract-loudness \
    --input_dir  "$AUDIO_DIR" \
    --output_dir "$LOUD_DIR" \
    --num_workers "$WORKERS"

echo "=== Step 3: Run inference ==="
soundaqnet-infer \
    --dataset_mel          "$MEL_DIR" \
    --dataset_wav_loudness "$LOUD_DIR"

echo "=== Step 4: Convert to DataFrame ==="
soundaqnet-to-df \
    --results_dir SoundAQnet_scene_ISOPl_ISOEv_PAQ8DAQs \
    --events_dir  SoundAQnet_event_probability \
    --output      soundaqnet_predictions.csv

echo "Done. Results in soundaqnet_predictions.csv"

---
## 6 · CLI vs Python API — when to use which

| Situation | Recommendation |
|---|---|
| Large dataset, scripted pipeline | **CLI** — easy to parallelize, resume, log |
| Interactive analysis, Jupyter | **Python API** — returns DataFrames directly |
| Single file or small batch | **Python API** `predict_from_audio` — fewest steps |
| Integrating into another codebase | **Python API** — importable, typed |
| HPC / cluster job | **CLI** — trivially scriptable, no Python interpreter overhead |

Both paths produce identical predictions.